# Minecraft RL Agent - Colab Training

Train your PPO agent on Google Colab's free T4 GPU.

**Two modes:**
- **Mock env** (default): Fake Minecraft with simulated rewards. Tests the full pipeline fast.
- **Craftium**: Real 3D Minecraft-like environment (install takes longer).

MineRL 0.4.4 has broken dependencies on modern Colab, so we skip it.

**Instructions:** Runtime -> Change runtime type -> T4 GPU -> Run all

In [ ]:
# ============================================================
# CONFIGURATION - Change these before running
# ============================================================
USE_CRAFTIUM = False   # Set True to use real 3D Minecraft-like env
TOTAL_STEPS = 1_000_000  # 1M steps ~ 2-3 hours on T4 with mock env

In [ ]:
# Step 1: Clone repo and install
import os

os.chdir('/content')
if not os.path.exists('/content/minecraft-ai'):
    !git clone https://github.com/Bobminion21/minecraft-ai.git
os.chdir('/content/minecraft-ai')

!pip install -q -e .

if USE_CRAFTIUM:
    !pip install -q craftium

print('Install complete.')

In [ ]:
# Step 2: Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/minecraft-ai'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/videos', exist_ok=True)
print(f'Checkpoints will save to: {DRIVE_DIR}/checkpoints')

In [ ]:
# Step 3: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Step 4: Configure and start training
from minecraft_ai.utils.config import Config
from minecraft_ai.training.trainer import Trainer

config = Config(
    total_timesteps=TOTAL_STEPS,
    rollout_length=128,
    ppo_epochs=4,
    num_minibatches=4,
    frame_size=64,
    frame_stack=4,
    max_episode_steps=2000,
    learning_rate=2.5e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_epsilon=0.1,
    entropy_coeff=0.01,
    checkpoint_interval=50_000,
    log_interval=5_000,
    checkpoint_dir=f'{DRIVE_DIR}/checkpoints',
    log_dir=f'{DRIVE_DIR}/logs',
    video_dir=f'{DRIVE_DIR}/videos',
    seed=42,
)

print(f'Training for {config.total_timesteps:,} steps')
print(f'Checkpoints every {config.checkpoint_interval:,} steps -> Google Drive')
print(f'Using: {"Craftium" if USE_CRAFTIUM else "Mock env"}')
print()

trainer = Trainer(config)  # Uses mock env by default
trainer.train()

In [ ]:
# Step 5: TensorBoard (run this cell, then click the link)
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/minecraft-ai/logs

In [ ]:
# Step 6: Plot training curves
from minecraft_ai.evaluation.plotting import plot_training_curves
import glob

csv_files = glob.glob(f'{DRIVE_DIR}/logs/*/metrics.csv')
if csv_files:
    plot_training_curves(csv_files[0], output_path=f'{DRIVE_DIR}/training_curves.png')
else:
    print('No metrics.csv found yet. Training needs to run longer.')

In [ ]:
# Step 7: Training summary
import numpy as np

print('=' * 50)
print('TRAINING SUMMARY')
print('=' * 50)
print(f'Total steps: {trainer.global_step:,}')
print(f'Episodes: {trainer.episode_count}')
if trainer.recent_rewards:
    print(f'Mean reward (last 100): {np.mean(trainer.recent_rewards[-100:]):.2f}')
    print(f'Best episode reward: {max(trainer.recent_rewards):.2f}')
print(f'\nCheckpoints saved to Google Drive.')
print(f'To resume: just Run All again -- it auto-loads the latest checkpoint.')